In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/544 Project/"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import random
import json
import time
from pathlib import Path
from collections import Counter
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm
from peft import PeftModel, PeftConfig
from sentence_transformers import SentenceTransformer, util as st_util

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ADD Hugging Face API KEY instead of HF_TOKEN
login(token="HF_TOKEN")

In [ ]:
MODEL_SOURCE = "sc-lora"
# ADD "Qwen/Qwen2.5-3B-Instruct" or "google/gemma-3-4b-it" instead of "MODEL_ID"
BASE_MODEL_ID = "MODEL_ID"

In [ ]:
# ADD "train/gemma/model_files/model" or "train/qwen/model_files/model" instead of "LORA_MODEL_PATH"
LORA_MODEL_PATH = "./PATH/TO/LORA/MODEL"
# ADD "train/gemma/model_files/tokenizer" or "train/qwen/model_files/tokenizer" instead of "LORA_TOKENIZER_PATH"
LORA_TOKENIZER_PATH = "./PATH/TO/LORA/TOKENIZER"

In [ ]:
DATA_PATHS = {
    "qa": "./data/test/halueval_qa_data.xlsx",
    "dialogue": "./data/test/halueval_dialogue_data.xlsx",
    "summarization": "./data/test/halueval_summarization_data.xlsx",
}

In [ ]:
# ADD "qa" or "dialogue" or "summarization" instead of "TASK"
TASK = "TASK"

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [ ]:
OUTPUT_PATH = f"./test_results/{BASE_MODEL_ID}/sc_results_{TASK}.jsonl"

In [ ]:
MAX_SAMPLES = None
MAX_NEW_TOKENS = 128
SC_N_SAMPLES = 3 # tried 3, 5 & 7; 3 gave the best speed/signal tradeoff on our GPU compute units
SC_TEMPERATURE = 0.8 # relatively stable
SC_TOP_P = 0.95
SC_DIVERGENCE_THRESHOLD = 0.35 # tried 0.2 - 0.6 on a dev subset; 0.35 gave the best F1 for hallucinated class
SC_ALPHA = 0.5
EMBEDDER_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [ ]:
def load_instruction(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [ ]:
INSTRUCTION_PATHS = {
    "qa": "./data/test/instruction_files/qa_evaluation_instruction.txt",
    "dialogue": "./data/test/instruction_files/dialogue_evaluation_instruction.txt",
    "summarization": "./data/test/instruction_files/summarization_evaluation_instruction.txt",
}

In [ ]:
INSTRUCTIONS = {}
print("Loading intruction files...")
for task, path in INSTRUCTION_PATHS.items():
    INSTRUCTIONS[task] = load_instruction(path)
    print(f"[{task}] Loaded from {path}  ({len(INSTRUCTIONS[task])} chars)")
print("Intruction files loaded")

In [ ]:
def build_chat_messages(task: str, row: dict, instruction: str) -> list:
    if task == "qa":
        user_content = (
            instruction
            + "\n\n#Question#: " + row["question"]
            + "\n#Answer#: " + row["answer"]
            + "\n#Your Judgement#:"
        )
    elif task == "dialogue":
        user_content = (
            instruction
            + "\n\n#Dialogue History#: " + row["dialogue_history"]
            + "\n#Response#: " + row["response"]
            + "\n#Your Judgement#:"
        )
    elif task == "summarization":
        user_content = (
            instruction
            + "\n\n#Document#: " + row["document"]
            + "\n#Summary#: " + row["summary"]
            + "\n#Your Judgement#:"
        )
    else:
        raise ValueError(f"Unknown task: {task}")
    return [{"role": "user", "content": user_content}]

In [ ]:
print(f"Reading base model from: {LORA_MODEL_PATH}/adapter_config.json")
peft_config = PeftConfig.from_pretrained(LORA_MODEL_PATH)
base_model_id = peft_config.base_model_name_or_path
print(f"\nbase model: {base_model_id}")
print(f"Loading base model weights\n")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
print(f"\nAttaching Lora adapter")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_PATH)
model = model.merge_and_unload()
print("lora weights merged")
model.eval()
print("Model ready")
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
print(f"Loading tokenizer from: {LORA_TOKENIZER_PATH}")
tokenizer = AutoTokenizer.from_pretrained(LORA_TOKENIZER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print("Tokenizer ready")

In [ ]:
print(f"Loading embedder: {EMBEDDER_NAME}\n")
embedder = SentenceTransformer(EMBEDDER_NAME)
print("\nEmbedder ready")

In [ ]:
def generate_samples(messages: list, n_samples: int = SC_N_SAMPLES) -> list:
  # single prompt -> batch of n samples identical inputs
    has_chat_template = (
        hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    )
    if has_chat_template:
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
    else:
        parts = []
        for m in messages:
            if "role" in m and m["role"] == "system" or m["role"] == "user":
                parts.append(m["content"])
        input_text = "\n\n".join(parts)

    inputs = tokenizer(input_text,return_tensors="pt",truncation=True,max_length=2048).to(DEVICE)
    input_ids = inputs["input_ids"].expand(n_samples, -1)
    attn_mask = inputs["attention_mask"].expand(n_samples, -1)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=SC_TEMPERATURE,
            top_p=SC_TOP_P,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)
    del inputs, output_ids
    torch.cuda.empty_cache()
    result = []
    for d in decoded:
        result.append(d.strip())
    return result

In [ ]:
def parse_judgement(raw: str) -> str:
    cleaned = raw.replace(".", "").strip()
    has_yes = "Yes" in cleaned
    has_no = "No"  in cleaned
    if (has_yes and has_no) or (not has_yes and not has_no):
        return "failed!"
    elif has_yes:
        return "Yes"
    else:
        return "No"

In [ ]:
def majority_vote(judgements: list) -> str:
  # out of n samples -> take the most common valid judgement
    valid = []
    for j in judgements:
        if j == "Yes" or j == "No":
            valid.append(j)
    if not valid:
        return "Fail"
    return Counter(valid).most_common(1)[0][0]

In [ ]:
def _token_f1(a: str, b: str) -> float:
    ta, tb = set(a.lower().split()), set(b.lower().split())
    if not ta or not tb:
        return 0.0
    common = ta & tb
    if not common:
        return 0.0
    p = len(common) / len(ta)
    r = len(common) / len(tb)
    return 2 * p * r / (p + r)

In [ ]:
def lexical_divergence(samples: list) -> float:
    if len(samples) < 2:
        return 0.0

    sims = []
    pairs = combinations(samples, 2)
    for pair in pairs:
        a = pair[0]
        b = pair[1]
        score = _token_f1(a, b)
        sims.append(score)

    total = 0.0
    count = 0
    for s in sims:
        total += s
        count += 1
    if count == 0:
        return 0.0
    mean_val = total / count
    return 1.0 - float(mean_val)

In [ ]:
def semantic_divergence(samples: list) -> float:
    if len(samples) < 2:
        return 0.0
    embs = embedder.encode(samples, convert_to_tensor=True, show_progress_bar=False)
    cosmat = st_util.cos_sim(embs, embs)
    n = len(samples)
    scores = []
    pairs = combinations(range(n), 2)
    for pair in pairs:
        i = pair[0]
        j = pair[1]
        value = cosmat[i][j].item()
        scores.append(value)
    return 1.0 - float(np.mean(scores))

In [ ]:
def combined_divergence(samples: list) -> dict:
    lex = lexical_divergence(samples)
    sem = semantic_divergence(samples)
    comb = SC_ALPHA * sem + (1 - SC_ALPHA) * lex
    return {"lexical_div": lex, "semantic_div": sem, "combined_div": comb}

In [ ]:
data_path = DATA_PATHS[TASK]
df = pd.read_excel(data_path)
print(f"Loaded {len(df)} rows from {data_path}")
print(df.columns.tolist())

In [ ]:
if MAX_SAMPLES is not None and MAX_SAMPLES < len(df):
    df = df.sample(MAX_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Subsampled to {len(df)} rows.")

In [ ]:
instruction = INSTRUCTIONS[TASK]
rng = random.Random(RANDOM_SEED)

In [ ]:
eval_rows = []
for _, row in df.iterrows():
    use_hallucinated = rng.random() > 0.5
    if TASK == "qa":
        base = {"knowledge": row["knowledge"], "question": row["question"]}
        if use_hallucinated:
            base["answer"] = row["hallucinated_answer"]
            base["ground_truth"] = "Yes"
        else:
            base["answer"] = row["right_answer"]
            base["ground_truth"] = "No"
    elif TASK == "dialogue":
        base = {"knowledge": row["knowledge"], "dialogue_history": row["dialogue_history"]}
        if use_hallucinated:
            base["response"] = row["hallucinated_response"]
            base["ground_truth"] = "Yes"
        else:
            base["response"] = row["right_response"]
            base["ground_truth"] = "No"
    elif TASK == "summarization":
        base = {"document": row["document"]}
        if use_hallucinated:
            base["summary"] = row["hallucinated_summary"]
            base["ground_truth"] = "Yes"
        else:
            base["summary"] = row["right_summary"]
            base["ground_truth"] = "No"
    eval_rows.append(base)

In [ ]:
yes_count = sum(1 for r in eval_rows if r["ground_truth"] == "Yes")
print(f"Evaluation set: {len(eval_rows)} samples | hallucinated={yes_count} correct={len(eval_rows)-yes_count}")

In [ ]:
output_file = Path(OUTPUT_PATH)
output_file.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
correct   = 0
incorrect = 0
failed    = 0
results   = []

In [ ]:
with output_file.open("w", encoding="utf-8") as fout:
    for i, row in enumerate(tqdm(eval_rows, desc=f"Evaluating SC [{TASK}]")):
        messages = build_chat_messages(TASK, row, instruction)

        try:
            raw_samples = generate_samples(messages, n_samples=SC_N_SAMPLES)
        except Exception as e:
            print(f"\nSample {i} ERROR: {e}")
            raw_samples = []

        if not raw_samples:
            judgement = "failed"
            parsed = []
            div_scores = {"lexical_div": None, "semantic_div": None, "combined_div": None}
            flagged = False
        else:
            parsed = [parse_judgement(r) for r in raw_samples]
            judgement = majority_vote(parsed)
            div_scores = combined_divergence(raw_samples)
            flagged = div_scores["combined_div"] >= SC_DIVERGENCE_THRESHOLD

        ground_truth = row["ground_truth"]

        if judgement == "failed":
            failed += 1
            incorrect += 1
        elif judgement == ground_truth:
            correct += 1
        else:
            incorrect += 1

        gen = {
            **row,
            "raw_output": raw_samples[0] if raw_samples else "",
            "all_samples": raw_samples,
            "all_parsed": parsed,
            "judgement": judgement,
            "lexical_div": div_scores["lexical_div"],
            "semantic_div": div_scores["semantic_div"],
            "combined_div": div_scores["combined_div"],
            "flagged": flagged,
        }
        results.append(gen)
        fout.write(json.dumps(gen, ensure_ascii=False) + "\n")
print(f"\nDone. {correct} correct | {incorrect} incorrect (incl. {failed} failed) | Total {len(eval_rows)}")

In [ ]:
valid = []
for r in results:
    if r["judgement"] != "failed":
        valid.append(r)
y_true = []
y_pred = []
for r in valid:
    y_true.append(r["ground_truth"])
    y_pred.append(r["judgement"])

total = len(results)
n_valid = len(valid)
n_fail = total - n_valid

if total > 0:
    acc_all = correct / total
else:
    acc_all = 0.0

if n_valid > 0:
    acc_valid = accuracy_score(y_true, y_pred)
else:
    acc_valid = 0.0

pos_label = "Yes"
prec = precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
rec = recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
f1 = f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)

In [ ]:
print(f"Task: {TASK}")
print(f"Base model: {base_model_id}")
# print(f"LoRA adapter: {LORA_MODEL_PATH}")
print(f"SC samples/input: {SC_N_SAMPLES}")
print(f"Total samples: {total}")
print(f"Valid (non-failed): {n_valid}")
print(f"Failed: {n_fail}")
print(f"Accuracy (all): {acc_all:.4f}")
print(f"Accuracy (valid): {acc_valid:.4f}")
print(f"Precision (Yes): {prec:.4f}")
print(f"Recall (Yes): {rec:.4f}")
print(f"F1 (Yes): {f1:.4f}")

In [ ]:
if n_valid > 0:
    print("\nClassification Report (valid samples):")
    print(classification_report(y_true, y_pred, digits=4))
    print("Confusion Matrix (rows=true, cols=pred):")
    labels = ["Yes", "No"]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{l}" for l in labels],
        columns=[f"Pred_{l}" for l in labels],
    )
    print(cm_df)

In [ ]:
res_df = pd.DataFrame(results)
valid_div = res_df.dropna(subset=["combined_div"])
print("\nDIVERGENCE SUMMARY\n")
print(f"Flagged as hallucinated: {valid_div['flagged'].sum()} / {len(valid_div)}")
print(f"Mean combined_div: {valid_div['combined_div'].mean():.4f}")
gt_yes = valid_div[valid_div["ground_truth"] == "Yes"]
gt_no = valid_div[valid_div["ground_truth"] == "No"]
if len(gt_yes):
    print(f"Mean div (hallu): {gt_yes['combined_div'].mean():.4f}")
if len(gt_no):
    print(f"Mean div (factual): {gt_no['combined_div'].mean():.4f}")

In [ ]:
summary = {
    "task": TASK,
    "model": base_model_id,
    "model_source": MODEL_SOURCE,
    # "lora_path": LORA_MODEL_PATH,
    "sc_n_samples": SC_N_SAMPLES,
    "total": total,
    "valid": n_valid,
    "failed": n_fail,
    "correct": correct,
    "incorrect": incorrect,
    "accuracy_all": round(acc_all, 4),
    "accuracy_valid": round(acc_valid, 4),
    "precision_yes": round(prec, 4),
    "recall_yes": round(rec, 4),
    "f1_yes": round(f1, 4),
}

In [ ]:
summary_path = "./test_results/results_benchmark_lora_selfconsistency.xlsx"
summary_df = pd.DataFrame([summary])
if Path(summary_path).exists():
    existing = pd.read_excel(summary_path)
    summary_df = pd.concat([existing, summary_df], ignore_index=True)
summary_df.to_excel(summary_path, index=False)
print(f"Summary saved/appended to {summary_path}")
summary_df